In [0]:
use catalog classic_workspace;
use schema default;

In [0]:
CREATE OR REPLACE TABLE products (
  product_id   INT     COMMENT 'Unique product identifier',
  name         STRING  COMMENT 'Product name',
  category     STRING  COMMENT 'Product category',
  price        DOUBLE  COMMENT 'Unit price in USD',
  in_stock     BOOLEAN COMMENT 'Whether the product is available'
)
USING DELTA
COMMENT 'Product catalogue — created with explicit DDL';

In [0]:
INSERT INTO products VALUES
  (1, 'Wireless Headphones', 'Electronics', 79.99,  true),
  (2, 'Standing Desk',       'Furniture',   349.00, true),
  (3, 'Mechanical Keyboard', 'Electronics', 129.50, false),
  (4, 'Monitor Stand',       'Furniture',   55.00,  true),
  (5, 'USB-C Hub',           'Electronics', 44.99,  true);

In [0]:
CREATE OR REPLACE TABLE electronics
USING DELTA
COMMENT 'Electronics subset — created with CTAS'
AS
SELECT product_id, name, price
FROM   products
WHERE  category = 'Electronics';

In [0]:
select * from electronics

In [0]:
describe electronics

In [0]:
%python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("order_id",   IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("customer",   StringType(),  False),
    StructField("quantity",   IntegerType(), False),
    StructField("total",      DoubleType(),  False),
])

data = [
    (101, 1, "Alice",   2, 159.98),
    (102, 2, "Bob",     1, 349.00),
    (103, 3, "Carol",   1, 129.50),
    (104, 5, "Alice",   3, 134.97),
    (105, 4, "David",   2, 110.00),
]

df = spark.createDataFrame(data, schema)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("classic_workspace.default.new_orders")

In [0]:
select * from new_orders

In [0]:
describe extended new_orders

In [0]:
CREATE OR REPLACE TABLE products_ext (
  product_id INT,
  name       STRING,
  price      DOUBLE
)
USING DELTA
LOCATION "s3://external-storage-bucket-624293007230-us-east-1-an/external-tables/products_ext"

In [0]:
INSERT INTO products_ext
SELECT * FROM electronics

In [0]:
describe extended products_ext

In [0]:
drop table products_ext

In [0]:
select * from products_ext